# SentencePiece

**Audience:** beginners who know BPE exists.

SentencePiece (Kudo & Richardson) is a tokenizer that treats the **raw Unicode string** as
the training unit. It does **not** have to split on spaces first. Spaces become a normal
character, drawn as `▁` (U+2581). That is why T5 and many multilingual models can handle
Chinese, Japanese, Thai, and English with one recipe.

Two inner models ship in the same library:

- **Unigram LM** (default) — start with a huge candidate vocab, then drop pieces that hurt
  likelihood the least.
- **BPE mode** — the same merge algorithm, but on the raw string (with `▁`), not on
  whitespace words.

We will build Unigram **intuition** (Viterbi segmentation) by hand, then train both modes
with `sentencepiece` **0.2.2** (pybind11 API).


## Learning path

```mermaid
flowchart LR
  raw[Raw text with spaces] --> mark["Spaces become ▁"]
  mark --> uni[Unigram: best segmentation]
  mark --> bpe[Optional BPE mode]
  uni --> lib[sentencepiece 0.2.2]
  bpe --> lib
```

> **Diagram tip.** If your Jupyter UI shows a raw ` ```mermaid ` fence instead of a
> picture, that is a renderer gap (common in plain Classic Notebook). Read the
> flowchart as text, or open the notebook on GitHub / VS Code.
>
> Full-page schematics (Token Lab):
> [diagrams gallery](https://sourangshupal.github.io/tokenization-explainer/diagrams/)
> · local `../site/diagrams/index.html`

> Schematic: [Detokenize marks (▁)](https://sourangshupal.github.io/tokenization-explainer/diagrams/05-detokenize-marks.html)
> · local [`../site/diagrams/05-detokenize-marks.html`](../site/diagrams/05-detokenize-marks.html)


## Why skip whitespace pre-tokenization?

English looks space-separated. Japanese and Chinese do not. If your first step is
`text.split()`, you have already baked in an English assumption.

SentencePiece's answer: the space is just another character. `"the cat"` is learned as
`▁the ▁cat` or `▁the ▁c at`, depending on the model. Decoding: every `▁` becomes a space.

```mermaid
flowchart TD
  s["the cat sat"] --> n["▁the ▁cat ▁sat"]
  n --> pieces["▁the / ▁cat / ▁sat"]
  pieces --> decode["the cat sat"]
```

> **Diagram tip.** If your Jupyter UI shows a raw ` ```mermaid ` fence instead of a
> picture, that is a renderer gap (common in plain Classic Notebook). Read the
> flowchart as text, or open the notebook on GitHub / VS Code.
>
> Full-page schematics (Token Lab):
> [diagrams gallery](https://sourangshupal.github.io/tokenization-explainer/diagrams/)
> · local `../site/diagrams/index.html`


## Setup


In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from IPython.display import display, Markdown
import ipywidgets as widgets

def find_root() -> Path:
    here = Path.cwd()
    for candidate in [here, here.parent]:
        if (candidate / "data" / "tiny_corpus.txt").exists():
            return candidate
    raise FileNotFoundError("Run the notebook from the repo root or the notebooks/ folder.")

ROOT = find_root()
CORPUS = ROOT / "data" / "tiny_corpus.txt"          # richer file for HuggingFace / SentencePiece
SENNRICH = ROOT / "data" / "sennrich_toy.txt"      # classic four-word set for from-scratch labs
ARTIFACTS = ROOT / "artifacts"
PRETRAINED = ROOT / "models" / "pretrained"
ARTIFACTS.mkdir(exist_ok=True)
print(f"course corpus: {CORPUS}")
print(f"Sennrich toy:  {SENNRICH}")
print()
print("Mermaid diagrams: GitHub and JupyterLab often render ```mermaid fences.")
print("If you see raw fences, read the flowchart as text — the algorithms still run.")


## Unigram intuition (no EM, just Viterbi)

A full Unigram trainer is an EM loop: guess piece probabilities, segment the corpus,
update probabilities, drop useless pieces. That is too much machinery for a first lesson.

The part you must feel: **given a piece vocabulary with scores, pick the segmentation
with the highest total score.** That search is Viterbi (dynamic programming).

Toy vocab for the string `tokenization`. Higher score = more likely piece.


In [ ]:
TOY_SCORES = {
    "t": -2.0,
    "o": -2.0,
    "k": -2.2,
    "e": -1.8,
    "n": -1.8,
    "i": -1.9,
    "z": -2.5,
    "a": -1.7,
    "token": -0.4,
    "tok": -0.9,
    "en": -1.1,
    "ization": -0.5,
    "ation": -0.8,
    "tion": -1.0,
}


def viterbi_segment(text: str, scores: dict[str, float]) -> list[str]:
    n = len(text)
    best = [float("-inf")] * (n + 1)
    back: list[int] = [-1] * (n + 1)
    chosen: list[str] = [""] * (n + 1)
    best[0] = 0.0
    for i in range(n):
        if best[i] == float("-inf"):
            continue
        for j in range(i + 1, n + 1):
            piece = text[i:j]
            if piece not in scores:
                continue
            cand = best[i] + scores[piece]
            if cand > best[j]:
                best[j] = cand
                back[j] = i
                chosen[j] = piece
    if best[n] == float("-inf"):
        raise ValueError(f"cannot segment {text!r} with this vocab")
    pieces: list[str] = []
    idx = n
    while idx > 0:
        pieces.append(chosen[idx])
        idx = back[idx]
    pieces.reverse()
    return pieces, best[n]


pieces, score = viterbi_segment("tokenization", TOY_SCORES)
print("best path:", pieces)
print("total score:", round(score, 3))


Try changing scores. If you make `"t"` extremely good and `"token"` terrible, Viterbi
will prefer a pile of single letters. Unigram training is mostly: **adjust those scores
so the corpus is cheap to encode, then delete pieces nobody needs.**


In [ ]:
box = widgets.Text(
    value="tokenization",
    description="Text:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "50px"},
)
out = widgets.Output()


def _run(_change=None) -> None:
    with out:
        out.clear_output()
        try:
            pieces, score = viterbi_segment(box.value, TOY_SCORES)
            print("pieces:", pieces)
            print("score: ", round(score, 3))
        except ValueError as exc:
            print(exc)
            print("Only characters listed in TOY_SCORES can be segmented in this toy.")


box.observe(_run, names="value")
_run()
display(box, out)


## Production library: `sentencepiece` 0.2.2

**API rules for 0.2.2 (pybind11 rewrite):**

- Use `return_type=` — **not** the deprecated alias `out_type`.
- Load models with `SentencePieceProcessor.from_file(...)`.
- Train with `SentencePieceTrainer.train(...)`.
- Do **not** call `EncodeAsImmutableProto` / `return_type='immutable_proto'` (removed).

We train two tiny models on `data/tiny_corpus.txt`: Unigram and BPE. Vocab 100 is enough
for this English file. `byte_fallback` is off so the vocab stays small; a later cell turns
it on for non-English text.

**Install fallback.** `sentencepiece` ships a native wheel. If `uv sync` fails on your
OS/Python combo, skip training and load
`models/pretrained/sp_unigram_tiny.model` (checked into the repo).


In [ ]:
import sentencepiece as spm

unigram_prefix = ARTIFACTS / "sp_unigram"
try:
    spm.SentencePieceTrainer.train(
        input=str(CORPUS),
        model_prefix=str(unigram_prefix),
        vocab_size=100,
        model_type="unigram",
        character_coverage=1.0,
        byte_fallback=False,
        minloglevel=1,
    )
    model_path = str(unigram_prefix) + ".model"
    print("trained:", model_path)
except Exception as exc:
    model_path = str(PRETRAINED / "sp_unigram_tiny.model")
    print("training failed — using checked-in model:", model_path)
    print("reason:", exc)

sp_uni = spm.SentencePieceProcessor.from_file(model_path)
print("unigram vocab", len(sp_uni))
print(sp_uni.encode("tokenization is the first step", return_type=str))
print(sp_uni.encode("tokenization is the first step", return_type=int))
print("round-trip:", sp_uni.decode(sp_uni.encode("tokenization is the first step")))
pieces = sp_uni.encode("the cat", return_type=str)
print("the cat →", pieces)
assert any("▁" in p for p in pieces), pieces
print("assert ok — space mark ▁ appears in pieces")


In [ ]:
bpe_prefix = ARTIFACTS / "sp_bpe"
try:
    spm.SentencePieceTrainer.train(
        input=str(CORPUS),
        model_prefix=str(bpe_prefix),
        vocab_size=100,
        model_type="bpe",
        character_coverage=1.0,
        byte_fallback=False,
        minloglevel=1,
    )
    sp_bpe = spm.SentencePieceProcessor.from_file(str(bpe_prefix) + ".model")
    print("bpe vocab", len(sp_bpe))
    print(sp_bpe.encode("tokenization is the first step", return_type=str))
except Exception as exc:
    print("BPE train skipped:", exc)
    print("Continue with sp_uni from the previous cell.")


## The `▁` mark and language-agnostic encoding

Look at the pieces. Almost every “word start” begins with `▁`. There is no separate
whitespace pre-tokenizer — the underscore-ish block is the space.


In [ ]:
samples = [
    "the cat sat",
    "tokenization",
    "unhappiness",
    "日本語",
    "বাংলা",
    "hello世界",
]
print(f"{'text':20} {'unigram':40} bpe")
for s in samples:
    u = " ".join(sp_uni.encode(s, return_type=str))
    b = " ".join(sp_bpe.encode(s, return_type=str))
    print(f"{s:20} {u:40} {b}")


Japanese and Bengali mostly become `[UNK]` or leftover characters here because the toy
corpus is English. Production multilingual SentencePiece models train on mixed text (or
enable byte fallback). Next cell trains a **byte-fallback** model so unknown scripts
become UTF-8 byte pieces like `<0xE0>` instead of a hard unknown.


In [ ]:
bf_prefix = ARTIFACTS / "sp_unigram_bytes"
spm.SentencePieceTrainer.train(
    input=str(CORPUS),
    model_prefix=str(bf_prefix),
    vocab_size=400,
    model_type="unigram",
    character_coverage=1.0,
    byte_fallback=True,
    minloglevel=1,
)
sp_bf = spm.SentencePieceProcessor.from_file(str(bf_prefix) + ".model")
for s in ["tokenization", "বাংলা", "日本語", "👋"]:
    print(s, "→", sp_bf.encode(s, return_type=str))


## Subword regularization (Unigram only)

Training is deterministic. **Encoding** can sample other valid segmentations. That noise
helps models generalize. `nbest_encode` lists the best paths. `enable_sampling=True`
draws one.

Use `return_type=str` (0.2.2 name). Do not pass `out_type`.


In [ ]:
text = "tokenization"
print("greedy:", sp_uni.encode(text, return_type=str))
print("nbest:")
for path in sp_uni.nbest_encode(text, nbest_size=5, return_type=str):
    print(" ", path)

print("samples:")
for _ in range(5):
    print(" ", sp_uni.encode(text, return_type=str, enable_sampling=True, alpha=0.1, nbest_size=-1))


## Interactive playground (trained Unigram model)


In [ ]:
box = widgets.Text(
    value="tokenization is the first step",
    description="Text:",
    layout=widgets.Layout(width="90%"),
    style={"description_width": "50px"},
)
out = widgets.Output()


def _run(_change=None) -> None:
    with out:
        out.clear_output()
        pieces = sp_uni.encode(box.value, return_type=str)
        ids = sp_uni.encode(box.value, return_type=int)
        print("pieces:", pieces)
        print("ids:   ", ids)
        print("decode:", sp_uni.decode(ids))


box.observe(_run, names="value")
_run()
display(box, out)


## Where you will see SentencePiece

- **T5**, mT5, ALBERT, XLNet — Unigram SentencePiece
- **Many Llama / Gemma / Mistral** releases ship a `.model` file from this library
- HuggingFace `tokenizers` can import those models, but training them is still
  `sentencepiece`

If a model card says `spiece.model` or `tokenizer.model`, it is almost certainly
SentencePiece.


## Exercises

1. Decode `sp_uni.encode("the cat", return_type=int)`. Where did the space go, and which
   piece carried it?
2. Compare Unigram vs BPE pieces for `unhappiness` on this tiny corpus. Which looks more
   stable if you change `vocab_size` to 80 and retrain?
3. Why did `বাংলা` need `byte_fallback=True` to avoid a wall of unknown characters?
4. Sampling: run the sampling cell twice. Why would a trainer want random segmentations
   during **model** training, but greedy segmentation at **inference**?

Golden notes: `INSTRUCTOR.md`. Capstone (model-card → algorithm) is in `01_bpe.ipynb`.
